In [1]:
import pandas as pd
import re

In [2]:
df = pd.read_excel(r'C:\Users\Pravasis Dhakal\Desktop\project code gravity\CG_Accommodation_expense.xlsx')

In [3]:
df

,APT,NEP/IND,2023-04-01 00:00:00,Unnamed: 3,Unnamed: 4,Unnamed: 5,2023-05-01 00:00:00,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 84,Unnamed: 85,2025-01-01 00:00:00,Unnamed: 87,Unnamed: 88,Unnamed: 89,2025-02-01 00:00:00,Unnamed: 91,Unnamed: 92,Unnamed: 93
0,NaN,NaN,Rent,Electricity,Internet,Atmos,Rent,Electricity,Internet,Atmos,...,Internet,Atmos,Rent,Electricity,Internet,Atmos,Rent,Electricity,Internet,Atmos
1,The Villas at Beavers Creek,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4142(Lease start 8/1/2023 - 7/31/2025) NEPALI,NEP,0,NaN,NaN,NaN,0,NaN,NaN,NaN,...,72.37,58.03,1815.92,146.95,79,164.7,1812.24,NaN,NaN,NaN
3,4064 (Lease start 8/14/2023 - 8/31/2025),IND,0,NaN,NaN,NaN,0,NaN,NaN,NaN,...,72.37,81.36,1770,151.27,79,120.95,1770,NaN,NaN,NaN
4,4076 (Lease start 10/13/2023 - 11/30/2025),IND,0,NaN,NaN,NaN,0,NaN,NaN,NaN,...,72.37,70,2020.92,150.49,79,NaN,2117.24,NaN,NaN,NaN
5,4050(10/6/2023 - 11/1/2024),NEP,0,NaN,NaN,NaN,0,NaN,NaN,NaN,...,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,4004 (Lease start 1/5/2024 - 1/31/2025),IND,0,NaN,NaN,NaN,0,NaN,NaN,NaN,...,72.37,300.32,1840.92,194.03,79,87.02,0,NaN,NaN,NaN
7,3088 (Lease start 1/22/2024 - 1/31/2025) NEPALI,NEP,0,NaN,NaN,NaN,0,NaN,NaN,NaN,...,72.37,151.41,1890.92,151.61,79,63.69,0,NaN,NaN,NaN
8,4084 (Lease start 7/1/2024 - 5/31/2025),IND,0,NaN,NaN,NaN,0,NaN,NaN,NaN,...,72.37,161.63,2442.7,160.56,80.41,584.55,2438.79,NaN,NaN,NaN
9,2156 (Lease start 10/1/2024 - 10/31/2025) NEPALI,NEP,0,NaN,NaN,NaN,0,NaN,NaN,NaN,...,72.37,36.28,2309.68,251.8,79,36.28,2338.9,NaN,NaN,NaN


In [4]:
def extract_apartment_info(value):
    """Extract apartment name, lease start date, and lease end date."""
    if pd.isna(value):
        return None, None, None
    match = re.search(r"^(.*?)\s*\((Lease start (\d{1,2}/\d{1,2}/\d{4}) - (\d{1,2}/\d{1,2}/\d{4}))?\)", value)
    if match:
        apartment_name = match.group(1).strip()
        lease_start = match.group(3) if match.group(3) else None
        lease_end = match.group(4) if match.group(4) else None
        return apartment_name, lease_start, lease_end
    return value, None, None

In [5]:
def extract_apartment_number(value):
    """Extract apartment number from the given value."""
    if pd.isna(value):
        return None
    match = re.search(r"(\d{3,5})", value)  # Extracts apartment number
    if match:
        return match.group(1)
    return None

In [6]:
def clean_accommodation_data(file_path):
    # Load the Excel file
    try:
        df = pd.read_excel(file_path, sheet_name=0)
    except FileNotFoundError:
        print("Error: File not found. Please check the file path.")
        return None

In [7]:
# Set the first row as column names and remove it from the data
df.columns =df.iloc[0]
df=df[1:].reset_index(drop=True)

In [8]:
# Rename columns to avoid unnamed issues
df.columns = [str(col) if not pd.isna(col) else f"Unnamed_{i}" for i, col in enumerate(df.columns)]

In [9]:
# Drop unnecessary rows with merged headers and reset index
df_cleaned = df.iloc[2:].reset_index(drop=True)

In [10]:
# Rename first column to 'Apartment' for clarity
df_cleaned.rename(columns={df_cleaned.columns[0]: 'Apartment'}, inplace=True)

In [11]:
# Forward-fill apartment names to ensure they align correctly with expenses
df_cleaned['Apartment'].fillna(method='ffill', inplace=True)

C:\Users\Pravasis Dhakal\AppData\Local\Temp\ipykernel_5228\2988680731.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned['Apartment'].fillna(method='ffill', inplace=True)


In [12]:
# Convert all column names to string to avoid iteration errors
df_cleaned.columns = df_cleaned.columns.astype(str)

In [13]:
# Identify numerical columns related to expenses
expense_columns = df_cleaned.columns[2:]

In [14]:
# Fill missing values in Electricity, Internet, and Atmos with column averages
for col in expense_columns:
    if "Electricity" in col or "Internet" in col or "Atmos" in col:
        df_cleaned[col].fillna(df_cleaned[col].mean(), inplace=True)

C:\Users\Pravasis Dhakal\AppData\Local\Temp\ipykernel_5228\3044713569.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned[col].fillna(df_cleaned[col].mean(), inplace=True)
C:\Users\Pravasis Dhakal\AppData\Local\Temp\ipykernel_5228\3044713569.py:4: FutureWarning: Dropping of nuisance columns in DataFrame reductions (with 'numeric_only=None') is deprecated; in a future version this will raise TypeError.  Select only valid columns before calling the reduction.
  df_cleaned[col].fillna(df_cleaned[col].mean(), inplace=True)


In [15]:
# Extract apartment details
df_cleaned[['Apartment', 'Lease Start Date', 'Lease End Date']] = df_cleaned['Apartment'].apply(
        lambda x: pd.Series(extract_apartment_info(x)))

In [16]:
# Categorize apartments
df_cleaned['Las Colinas'] = df_cleaned['Apartment'].apply(lambda x: 'Yes' if 'Las Colinas' in str(x) else 'No')
df_cleaned['The Villas at Beavers Creek'] = df_cleaned['Apartment'].apply(lambda x: 'Yes' if 'Villas at Beavers Creek' in str(x) else 'No')

In [17]:
# Extract apartment numbers
df_cleaned['Apartment Number'] = df_cleaned['Apartment'].apply(extract_apartment_number)


In [18]:
# Select relevant columns for the final dataset
cleaned_df = df_cleaned[['Apartment', 'Apartment Number', 'Las Colinas', 'The Villas at Beavers Creek',
                             'Lease Start Date', 'Lease End Date'] + list(expense_columns)]


In [19]:
def save_cleaned_data(df, output_path="Structured_CG_Accommodation_Expense_cg.xlsx"):
    """Saves the structured data to an Excel file."""
    df.to_excel(output_path, index=False)
    print(f"Data cleaning and transformation completed. Saved as '{output_path}'.")

In [20]:
cleaned_df


,Apartment,Apartment Number,Las Colinas,The Villas at Beavers Creek,Lease Start Date,Lease End Date,Rent,Rent,Rent,Rent,...,Atmos,Atmos,Atmos,Atmos,Atmos,Atmos,Atmos,Atmos,Atmos,Atmos
0,4064,4064,No,No,8/14/2023,8/31/2025,0,0,0,0,...,103.6,81.36,79.83,91.69,91.69,120.95,123.56,81.36,120.95,NaN
1,4076,4076,No,No,10/13/2023,11/30/2025,0,0,0,0,...,95.68,76.35,78.3,92.35,90.5,85,62,70,NaN,NaN
2,4050(10/6/2023 - 11/1/2024),4050,No,No,None,None,0,0,0,0,...,170.79,300.32,114.78,33.85,50.07,87.02,0,0,NaN,NaN
3,4004,4004,No,No,1/5/2024,1/31/2025,0,0,0,0,...,408.31,151.41,138.09,25.13,25.12,63.69,345.67,300.32,87.02,NaN
4,3088,3088,No,No,1/22/2024,1/31/2025,0,0,0,0,...,111.41,161.63,52.14,360.42,41.6,584.55,220.37,151.41,63.69,NaN
5,4084,4084,No,No,7/1/2024,5/31/2025,0,0,0,0,...,0,0,184.38,208.83,397.88,36.28,120.53,161.63,584.55,NaN
6,2156,2156,No,No,10/1/2024,10/31/2025,0,0,0,0,...,0,0,0,125.11,0,0,98.35,36.28,36.28,NaN
7,None,None,No,No,None,None,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,Las Colinas,None,Yes,No,None,None,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,3877 (7/1/2024 - 6/30/2024),3877,No,No,None,None,1868.88,1868.88,1868.88,1868.88,...,0,0,0,0,0,0,0,0,0,NaN
